# Performance: Parameterization Time vs Molecule Size (Linear Alkanes)

Benchmark wall time for Interchange vs OMMFFS parameterization of linear alkanes of increasing size.

In [1]:
import signal
import time

import matplotlib.pyplot as plt
import numpy as np
import openmm
from openff.toolkit import ForceField as OFFForceField
from openff.toolkit import Molecule
from openmm.app import ForceField, NoCutoff

from openmmforcefields.generators import SMIRNOFFTemplateGenerator

In [2]:
FF_FILE = "openff_no_water-3.0.0-alpha0.offxml"
#SIZES = [5, 10, 20, 50, 100, 200, 400, 600, 800, 1000]
SIZES = [200, 400, 600, 800, 1000]
N_REPEATS = 3
TIMEOUT = 60  # seconds


class TimeoutError(Exception):
    pass


def _timeout_handler(signum, frame):
    raise TimeoutError("Timed out")


def time_interchange(mol, ff_file):
    """Time Interchange parameterization."""
    ff = OFFForceField(ff_file)
    top = mol.to_topology()
    start = time.perf_counter()
    ff.create_openmm_system(top)
    return time.perf_counter() - start


def time_ommffs(mol, ff_file):
    """Time OMMFFS template generator parameterization."""
    generator = SMIRNOFFTemplateGenerator(molecules=[mol], forcefield=ff_file)
    openmm_ff = ForceField()
    openmm_ff.registerTemplateGenerator(generator.generator)
    omm_top = mol.to_topology().to_openmm()
    start = time.perf_counter()
    openmm_ff.createSystem(omm_top, nonbondedMethod=NoCutoff)
    return time.perf_counter() - start


def benchmark_with_timeout(func, mol, ff_file, timeout_s):
    """Run a benchmark with a timeout. Returns time or None if timed out."""
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(timeout_s)
    try:
        result = func(mol, ff_file)
        signal.alarm(0)
        return result
    except TimeoutError:
        return None
    finally:
        signal.signal(signal.SIGALRM, old_handler)

In [4]:
results = {"n_carbons": [], "n_heavy": [], "interchange": [], "ommffs": []}

for n in SIZES:
    smiles = "C" * n
    mol = Molecule.from_smiles(smiles)
    #mol.generate_conformers(n_conformers=1)
    n_heavy = sum(1 for atom in mol.atoms if atom.atomic_number != 1)

    print(f"\nBenchmarking n={n} carbons ({n_heavy} heavy atoms, {mol.n_atoms} total)...")

    # Interchange timing
    ic_times = []
    for rep in range(N_REPEATS):
        t = benchmark_with_timeout(time_interchange, mol, FF_FILE, TIMEOUT)
        if t is None:
            print(f"  Interchange timed out at {TIMEOUT}s")
            break
        ic_times.append(t)

    # OMMFFS timing
    of_times = []
    for rep in range(N_REPEATS):
        t = benchmark_with_timeout(time_ommffs, mol, FF_FILE, TIMEOUT)
        if t is None:
            print(f"  OMMFFS timed out at {TIMEOUT}s")
            break
        of_times.append(t)

    ic_mean = np.mean(ic_times) if ic_times else None
    of_mean = np.mean(of_times) if of_times else None

    results["n_carbons"].append(n)
    results["n_heavy"].append(n_heavy)
    results["interchange"].append(ic_mean)
    results["ommffs"].append(of_mean)

    ic_str = f"{ic_mean:.3f}s" if ic_mean else "TIMEOUT"
    of_str = f"{of_mean:.3f}s" if of_mean else "TIMEOUT"
    speedup = f"{ic_mean / of_mean:.1f}x" if (ic_mean and of_mean) else "N/A"
    print(f"  Interchange: {ic_str}, OMMFFS: {of_str}, Speedup: {speedup}")


Benchmarking n=200 carbons (200 heavy atoms, 602 total)...
  Interchange: 1.928s, OMMFFS: 1.966s, Speedup: 1.0x

Benchmarking n=400 carbons (400 heavy atoms, 1202 total)...
  Interchange: 2.560s, OMMFFS: 3.506s, Speedup: 0.7x

Benchmarking n=600 carbons (600 heavy atoms, 1802 total)...
  Interchange: 3.417s, OMMFFS: 5.649s, Speedup: 0.6x

Benchmarking n=800 carbons (800 heavy atoms, 2402 total)...
  Interchange: 4.903s, OMMFFS: 8.052s, Speedup: 0.6x

Benchmarking n=1000 carbons (1000 heavy atoms, 3002 total)...
  Interchange: 6.193s, OMMFFS: 11.866s, Speedup: 0.5x


In [ ]:
# Filter out None values for plotting
valid = [(nh, ic, of) for nh, ic, of in zip(results["n_heavy"], results["interchange"], results["ommffs"])
         if ic is not None and of is not None]

if valid:
    n_heavy_valid, ic_valid, of_valid = zip(*valid)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Log-log timing plot
    ax1.loglog(n_heavy_valid, ic_valid, "o-", label="Interchange", color="tab:blue")
    ax1.loglog(n_heavy_valid, of_valid, "s-", label="OMMFFS", color="tab:orange")
    ax1.set_xlabel("Number of heavy atoms")
    ax1.set_ylabel("Wall time (s)")
    ax1.set_title("Parameterization Time: Linear Alkanes")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Speedup ratio plot
    speedups = [ic / of for ic, of in zip(ic_valid, of_valid)]
    ax2.semilogx(n_heavy_valid, speedups, "D-", color="tab:green")
    ax2.axhline(y=1, color="gray", linestyle="--", alpha=0.5)
    ax2.set_xlabel("Number of heavy atoms")
    ax2.set_ylabel("Speedup (Interchange / OMMFFS)")
    ax2.set_title("Speedup Ratio")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("linear_alkane_performance.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No valid data to plot (all timed out)")

In [ ]:
# Summary table
print(f"{'N carbons':>10} {'N heavy':>8} {'Interchange (s)':>16} {'OMMFFS (s)':>12} {'Speedup':>10}")
print("-" * 58)
for nc, nh, ic, of in zip(results["n_carbons"], results["n_heavy"], results["interchange"], results["ommffs"]):
    ic_str = f"{ic:.4f}" if ic else "TIMEOUT"
    of_str = f"{of:.4f}" if of else "TIMEOUT"
    sp = f"{ic/of:.1f}x" if (ic and of) else "N/A"
    print(f"{nc:10d} {nh:8d} {ic_str:>16} {of_str:>12} {sp:>10}")